# Khmer OCR — Predict a folder with `vgg_bilstm_ctc_10k`

Uses this repo's `final/predict.py`, which rebuilds the model from its own
`experiment.json` config and loads the charset it was trained on — the right tool
for runs under `outputs_crnn/`.

**How to use:** drop images into the `images/` folder next to this notebook, then run
the cell below. Every image (png/jpg/jpeg/bmp/tif/webp, searched recursively) gets
predicted, printed inline, and saved to `output.txt`.

In [3]:
import importlib.util
import sys
from pathlib import Path

import torch

# ===================== EDIT THESE =====================
IMAGES_DIR = "/home/thareah/Pictures/test"          # folder of images to predict
# Either a bare run name ("vgg_bilstm_ctc_10k") OR a path to the run folder
# ("../outputs_crnn/vgg_bilstm_ctc_10k") — only the folder name is used to match.
RUN_NAME   = "../outputs_crnn/vgg_bilstm_ctc_50k"
# ======================================================

run_key = Path(RUN_NAME).name  # normalize a path down to just the run folder name

# Load THIS repo's final/predict.py by explicit path (avoids any stale cached
# `predict` module in the kernel). Works whether cwd is final/ or the repo root.
FINAL_DIR = Path.cwd() if (Path.cwd() / "predict.py").exists() else Path.cwd() / "final"
sys.path.insert(0, str(FINAL_DIR))           # so predict.py can import src/
_spec = importlib.util.spec_from_file_location("final_predict", FINAL_DIR / "predict.py")
P = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(P)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Outputs root:", P.DEFAULT_OUTPUTS_ROOT)

# Build just the one run (it rebuilds from experiment.json + its own training charset).
predictors = P.load_predictors(
    P.DEFAULT_OUTPUTS_ROOT, P.DEFAULT_CHARSET, device, only=[run_key]
)
available = [r[0] for r in P.discover_runs(P.DEFAULT_OUTPUTS_ROOT)]
assert predictors, f"Run '{run_key}' not found. Available: {available}"
predictor = predictors[0]
print(f"Using {predictor.run_name} ({predictor.arch})\n")

# Predict every image in the folder.
Path(IMAGES_DIR).mkdir(parents=True, exist_ok=True)
images = P.gather_images([IMAGES_DIR])
print(f"Found {len(images)} image(s) in '{IMAGES_DIR}'\n")

lines = []
for img in images:
    try:
        text = predictor.predict(img)
    except Exception as e:
        text = f"<error: {type(e).__name__}: {e}>"
    line = f"{img.name}\n  -> {text}"
    print(line + "\n")
    lines.append(line)

Path("output.txt").write_text("\n".join(lines), encoding="utf-8")
print("Saved -> output.txt")

Device: cuda
Outputs root: /home/thareah/Desktop/server_config/reah_ocr/traditional/outputs_crnn
  loaded  vgg_bilstm_ctc_50k               (vgg_bilstm_ctc)
Using vgg_bilstm_ctc_50k (vgg_bilstm_ctc)

Found 15 image(s) in '/home/thareah/Pictures/test'

Screenshot From 2026-05-27 13-48-03.png
  ->  ការចិញ្ចឹមមាន់នៅក្នុងប្រទេសកម្ពុជាយើងនេះ

Screenshot From 2026-05-27 13-48-12.png
  ->  កសិករភាគច្រើនចិញ្ចឹមមាន់ក្នុងស្រុក

Screenshot From 2026-05-27 13-48-17.png
  ->  ចំពោះមាន់បរទេសមាន

Screenshot From 2026-05-27 13-48-28.png
  -> ការលូតលាស់លឿនមិនសូវមានអ្នកចិញ្ចឹម

Screenshot From 2026-05-27 13-49-14.png
  -> ដោយសារតែគ្មានកន្លែងផលិតពូជនៅក្នុងប្រទេស

Screenshot From 2026-05-27 13-49-18.png
  -> និងមានតំលៃខ្ពស់ព្រោះ

Screenshot From 2026-05-27 13-49-27.png
  ->  នាំចូលពីបរទេស ។

Screenshot From 2026-05-27 13-49-56.png
  -> ធម្មជាតិ អ៊ី អ៊ិក្ស អិម ។

Screenshot From 2026-05-27 13-50-02.png
  -> ធម្មជាតិ អ៊ីអ៊ិក្សអិម។

Screenshot From 2026-06-01 13-33-16.png
  -> ច្បាប់ភូមិបាល២០០១ របស់ប្រទេសកម្